### This notebook will contain my notes from the COMP 579 textbook "Reinforcement Learning - an Introduction (2nd edition)" by Richard S. Sutton and Andrew G. Barto

Chapters worth looking at :
- 3.6 (Optimal policies and Value Functions)
- 4.1 (Policy Evaluation)
- 4.2 (Policy Improvement)
- 4.3 (Policy Iteration)
- 4.4 (Value Iteration)

- 13.5 (Actor Critic Methods)

"Reinforcement learning is learning what to do—how to map situations to actions—so as to maximize a numerical reward signal"

Markov Decision Processes (MDP) incorporate the three pillars of ML :
1. That an agent must be able to _sense the state of its environment_ to some extent, and ;
2. Must be able to _take actions that affect the state_, and ;
3. Must have _a goal / goals_ relating to the states of the environment

Reinforcement Learning is not supervised learning ; and not unsupervised learning either

Reinforcement Learning leverages the "exploitation vs exploration" dilemna.

The policy $\pi(s,a)$ is a mapping from states to actions to be taken when in those states

Whereas the reward signal indicates what is good in an immediate sense, a value function specifies what is good in the long run. Roughly speaking, the value of a state is the total amount of reward an agent can expect to accumulate over the future, starting from that state.

So the value of a particular state defines the long-term return from starting in that state and following our policy.

Notes from Chapter  3 - "Finite Markov Decision Processes"

$(S_t) \xrightarrow{\text{A}_t} (R_{t+1}, S_{t+1})$

The _trajectory_ of an agent looks like : $S_0,A_0,R_1,S_1,A_1,R_2,S_2...$

$G \doteq \sum_0^T R_t$ where T corresponds to the **time of termination** (at a terminal state)

Discounted return $G_t \doteq \sum_{k=0}^\infty \gamma^k R_{t+k+1} = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ...$

Q : Does discounted return addresses the problem of "continuing tasks" i.e when nthere is no terminal task, and the learning process goes on forever ?

$$G_t \doteq R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ...$$
$$=R_{t+1} + \gamma (R_{t+2} + \gamma R_{t+3} + ...)$$
$$=R_{t+1} + \gamma G_{t+1}$$

This is the recurrent definition of discounted return.

## 3.5 - Policies and value functions

$\pi(a|s)$ is the probability of taking action in a given state s

The value function of a state s under policy $\pi$, denoted $V_{\pi}(s)$ is the expected return when starting in s and following $\pi$ thereafter

$V_{\pi}(s) \doteq \mathbb{E}\left( G_t | S_t = s\right) = \mathbb{E}\left( \sum_{k=0}^\infty \gamma^k R_{t+k+1} | S_t = s\right)$

Similarily we define the _action value function for policy $\pi$_

$q_\pi(s,a) = \mathbb{E}\left( G_t | S_t = s, A_t = a\right)$

Exercise : show that $V_\pi(s)= \sum_a \pi(s|a) \sum_{s',r} p(s',r | s,a) (r + \gamma V_\pi(s'))$

We use the recursion of discounted return :
$V_{\pi}(s) \doteq \mathbb{E}\left( G_t | S_t = s\right) = \mathbb{E}\left( R_{t+1} + \gamma G_{t+1} | S_t = s\right) = \mathbb{E}\left( R_{t+1}\right | S_t = s) + \gamma \mathbb{E}\left( G_{t+1} | S_t = s\right)$

We use the law of total expectation which states $E[X] = \sum_y E(X | Y = y)p(Y = y)$. Intuitively we break down the expectation of X into sub-expectations that are weighted by their probability.

$\mathbb{E}\left( R_{t+1}\right | S_t = s)= \sum_{a} \pi(a|s) \mathbb{E}(R_{t+1} | S_t=s,A_t=a)$


Similarly, applying the law of total expectation to the future return:

$\mathbb{E}(G_{t+1}|S_t=s)=\sum_a\pi(a|s)\mathbb{E}(G_{t+1}|S_t=s,A_t=a)$

Substituting both expressions into the original equation:

$V_\pi(s)=\sum_a\pi(a|s)\left[\mathbb{E}(R_{t+1}|s,a)+\gamma\mathbb{E}(G_{t+1}|s,a)\right]$

We apply the law of total expectation again, this time conditioning on the possible next states $s'$ and rewards $r$, whose joint probabilities are given by $p(s',r|s,a)$:

$V_\pi(s)=\sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)\mathbb{E}(R_{t+1}+\gamma G_{t+1}|s,a,s',r)$

Since we condition on the reward $R_{t+1}=r$, it is no longer random:

$V_\pi(s)=\sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)\left[r+\gamma\mathbb{E}(G_{t+1}|s,a,s',r)\right]$

By the Markov property, the expected future return depends only on the next state $s'$ (assuming we continue following policy $\pi$):

$\mathbb{E}(G_{t+1}|s,a,s',r)=\mathbb{E}(G_{t+1}|S_{t+1}=s')=V_\pi(s')$

Therefore, we obtain the **Bellman expectation equation**:

$V_\pi(s)=\sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)\left[r+\gamma V_\pi(s')\right]$

**Interpretation:** The value of a state is the probability-weighted average of the immediate reward plus the discounted value of the next state, accounting for both the stochastic policy and the environment's transitions.


## Optimal policies and Optimal value functions

_Policy evaluation_ consists in computing the state-value function $v_\pi(s)$ through interaction with the environment

Iterative policy evaluation uses the fact that the state-value function looks like $v_\pi(S_t) = f(v_\pi(S_{t+1}))$

The agent is at step $s_0$. We wish to determine the value function for that state.
We make a guess about $v_\pi(s_1)$ then update $v_\pi(s_0)$.

The agent is not involved in this process right ? We simply iterate over all the states and "peek" at the states that would have been chosen by our policy ?



In [ ]:
#value Iteration improves the policy at every state, whereas policy iteration performs multiple sweeps

def value_iteration(gamma=0.99):

    for s in grid_coords:
        q_values = {}

        for action in possible_actions:
            s_next = apply_policy(s, action)
            reward = get_reward(s_next)

            q_values[action] = reward + gamma * value[s_next]

        value[s] = max(q_values.values())
    # this is very similar to the policy iteration, however here instead of returning a new policy we make value greedy w.r.t the q-value function.


In [ ]:
# Exercise to apply iterative policy evaluation : the gridworld example
# We will estimate the value of each state of the 4x4 grid with iterative policy evaluation

import numpy as np

grid = np.array([
[-1,0,0,1],
[0,1,0,0],
[1,0,0,0],
[0,-1,0,1]
])


grid_coords = [(x, y) for x in range(grid.shape[0]) for y in range(grid.shape[1])]
value = {state:0 for state in grid_coords}

import numpy as np

possible_actions = [
    (-1, 0),  # Up
    (1, 0),   # Down
    (0, -1),  # Left
    (0, 1)    # Right
]

def apply_policy(s: tuple,action):

    # Calculate candidate next state
    s_next = (s[0] + action[0], s[1] + action[1])

    # Check whether the candidate is within the grid
    if 0 <= s_next[0] < grid.shape[0] and 0 <= s_next[1] < grid.shape[1]:
        return s_next

    # Invalid move: remain in the current state
    return s

def get_reward(s : tuple):
    return grid[*s]

def value_update(s, V, gamma=0.9):
    new_value = 0

    for action in possible_actions:
        s_next = apply_policy(s, action)
        reward = get_reward(s_next)

        new_value += 0.25 * (reward + gamma * V[s_next])

    return new_value


def iterative_policy_update():
    print("initial state-value function",value)
    for i in range(10000):
        for state in grid_coords:
            value[state] = value_update(state, value)

    print("state-value function after complete sweep with 10 iterations",value)

iterative_policy_update()

In [36]:
import numpy as np
import plotly.express as px

v = np.array(list(value.values())).reshape(grid.shape) # reshape into the original grid array

fig = px.imshow(
    v,
    text_auto=".2f",
    title="State-Value Function"
)

fig.update_layout(
    xaxis_title="Column",
    yaxis_title="Row"
)

fig.show()

**Iterative policy evaluation works as follows:**

1. Initialize the state-value function estimates $V(s)$ to arbitrary values for every state $s \in \mathcal{S}$ (here, a tabular representation for the 4×4 grid), with terminal-state values fixed at zero if applicable.
2. For each nonterminal state $s$, update its estimated value using the Bellman expectation equation, considering all possible actions $a$ under policy $\pi$, their associated rewards $r$, and possible successor states $s'$.
3. Repeat step 2 over multiple iterations (called *sweeps*) until the state-value estimates converge to $v_\pi(s)$, the true state-value function **under the fixed policy $\pi$**, which is not necessarily the optimal state-value function $v_*(s)$.

**Note on transition probabilities:**

The policy assigns equal probability $\pi(a|s)=1/4$ to each of the four actions (up, down, left, right), regardless of the current state.

However, the transition probabilities $p(s',r|s,a)$ describe the environment's response to a selected action, rather than the probability of selecting that action.

In our **deterministic environment**, each state-action pair $(s,a)$ produces exactly one successor-state/reward pair $(s',r)$. Therefore, $p(s',r|s,a)=1$ for that particular outcome and $0$ for every other outcome.

For example, attempting to move up from the top-left corner $s=(0,0)$ is an invalid movement. However, the agent remains in its current state with certainty:

$p(s'=(0,0),r=r_{\text{wall}}|s=(0,0),a=\text{up})=1$

where $r_{\text{wall}}$ is the reward assigned to that transition.

Thus, an invalid movement does not imply a transition probability of zero for every outcome. Instead, the environment deterministically returns the agent to its current state.

In a **stochastic environment**, the same state-action pair could produce several possible successor-state/reward pairs, each with its own probability. The Bellman expectation equation would then average over these outcomes, weighted by their transition probabilities.

DP methods assume complete knowledge of the transition probabilities

Key idea of DP : use state-value function $V_{\pi}(s)$ to organise anbd structure the search for good policies.

The optimal policy always chooses the action that maximizes the expected
return, assuming optimal behavior thereafter. Therefore, the
**Bellman optimality equation** for the state-value function is:

$$
v_*(s)=\max_a \mathbb{E}\left[
R_{t+1}+\gamma v_*(S_{t+1})
\mid S_t=s,A_t=a
\right]
$$

Here, $\max_a$ means taking the maximum **over all possible actions**
$a$ available in state $s$.

For each action, we compute the expected immediate reward plus the
discounted optimal value of the next state, then select the maximum.

"All the updates done in DP algorithms are called expected updates because they are based on an expectation over all possible next states rather than on a sample next state."

DP examines all possible actions then takes the one that maximises the value function, hence the $\max_a$. This distinguishes **expected update** from a **sample update**.


**Policy Evaluation** is about computing the state value function

$$v_\pi(s) = \mathbb{E_\pi} [G_t | S_t = s]$$

We make an initial gess about $v_\pi(s)$ then update it as follows :

$$v_{k+1} \leftarrow \sum_{a} \pi(a|s) \sum_{s',r} p(s\ ,r | s,a) [r + \gamma v_k(s\ )]$$

This algorithm makes a sequence which converges to $v_{\pi}(s)$ and is called _iterative policy iteration_

## Policy Improvement

* $v_\pi(s)$ represents the expected return starting from state $s$ and following policy $\pi$ immediately and thereafter.

* $q_\pi(s,a)$ represents the expected return starting from state $s$, taking action $a$ first, and then following policy $\pi$ thereafter.

This allows us to evaluate whether selecting a particular action instead of following the current policy would improve the expected return.

One way to improve the policy is to identify an action $a$ such that $q_\pi(s,a)\geq v_\pi(s)$. This means that taking action $a$ and then following the original policy $\pi$ yields an expected return at least as high as following $\pi$ from the beginning. A strict inequality indicates an improvement.

Therefore, we can construct an improved policy by selecting the action that maximizes the action-value function:

$\pi'(s)=\arg\max_a q_\pi(s,a)$

In other words, we make the new policy **greedy with respect to the action-value function $q_\pi$ _of the original policy_**.

According to the **policy improvement theorem**, if the new policy satisfies $q_\pi(s,\pi'(s))\geq v_\pi(s)$ for every state $s$, then:

$v_{\pi'}(s)\geq v_\pi(s),\quad \forall s\in\mathcal{S}$

Thus, the improved policy is guaranteed to perform at least as well as the original policy in every state.

## Policy Iteration

Policy Iteration = Policy Evaluation (with policy iterative evaluation for ex) + Policy Improvement

We stop the sequential process of evaluation / improvement when the policy is said to be _stable_, i.e when it does not differ from the Q-function. At that point, $\pi(s)=\pi^*(s)$ which means we reached optimality.

$q_\pi(s,a)=\sum_{s',r}p(s',r|s,a)\mathbb{E}(R_{t+1}+\gamma G_{t+1}|s,a,s',r) = R(s,a) + \gamma V(s')$

Policy iteration consists of two simultaneous, interacting processes, one making the value function consistent with the current policy (policy evaluation), and the other making the policy greedy with respect to the current value function (policy improvement). In policy iteration, these two processes alternate, each completing before the other begins

In [39]:
# let's apply policy iteration to our previous example
# in the policy improvement phase, we compute q(s,a) for the states then compare that to the value that was stored before

def policy_iteration(gamma=0.99):
    new_policy = {}

    for s in grid_coords:
        q_values = {}

        for action in possible_actions:
            s_next = apply_policy(s, action)
            reward = get_reward(s_next)

            q_values[action] = reward + gamma * value[s_next]
            #updating the state-action function

        # Choose action maximizing q_pi(s,a)
        best_action = max(q_values, key=q_values.get)
        new_policy[s] = best_action

    return new_policy

print(policy_iteration())

{(0, 0): (1, 0), (0, 1): (1, 0), (0, 2): (0, 1), (0, 3): (-1, 0), (1, 0): (1, 0), (1, 1): (0, 1), (1, 2): (0, 1), (1, 3): (-1, 0), (2, 0): (0, -1), (2, 1): (0, -1), (2, 2): (0, 1), (2, 3): (1, 0), (3, 0): (-1, 0), (3, 1): (0, 1), (3, 2): (0, 1), (3, 3): (1, 0)}


## Important distinction :

`max` $\ne$ `argmax`!!

When we write $v_{k+1}= \underset{a}{\max} \mathbb{E}\left[ R_{t+1} + \gamma v_k(S_{t+1}) | S_t=s, A_t = a \right]$

We mean that we want to find the maximum of $$\mathbb{E}\left[ R_{t+1} + \gamma v_k(S_{t+1}) | S_t=s, A_t = a \right]$$ for a GIVEN s (fixed), _but letting a vary._





## Chapter 5.5.  : Off-policy prediction (evaluation) via Importance Sampling

RL algorithms have to balance exploitation (optimal policy) with exploration (non optimal policy).

The agent learns action values not from the optimal policy (which would make its actions greedy wrt value function) but from a less optimal policy (exploration) which ensures exploration of the state-action space.

$$\text{behavior policy} \rightarrow \text{target policy}$$

In on-policy prediction, the agent's learning policy (the target) is the same as the behaving policy (which estimates the value function)

In general the target policy is deterministic, whereas the behavior policy is stochastic, in order to favor exploration.




## Chapter 6 - Temporal difference learning

Recall that policy evaluation consists in estimating the value function $v_\pi$ for a given policy $\pi$.

MC methods don't need knowledge of the transition probabilities but need to go through one full episode before updating the policy.

DP methods on the other hand need knowledge of the transition probabilities (they "peek" at future states) but can update the policy after a single time step.

**Temporal-Difference (TD) learning** combines ideas from Monte Carlo (MC) and Dynamic Programming (DP). Like MC, it learns from sampled interactions with the environment without requiring a model. Like DP, it uses *bootstrapping*, updating a state-value estimate using the estimated value of a successor state.

We define the TD(0) update as:

$$V(S_t)\leftarrow V(S_t)+\alpha\left[R_{t+1}+\gamma V(S_{t+1})-V(S_t)\right]$$

We want to estimate the true state-value function $v_\pi(s)$ using our current approximation $V(s)$.

After following policy $\pi$, we observe a transition from state $S_t$ to state $S_{t+1}$, receiving reward $R_{t+1}$.

TD learning tries to satisfy the Bellman expectation equation using _individual sampled transitions_ rather than _explicitly calculating the expectation over all possible next state_.

We define the **TD target**:

$$Y_t=R_{t+1}+\gamma V(S_{t+1})$$

Recall that the Bellman expectation equation states:

$$v_\pi(s)=\mathbb{E}_\pi[R_{t+1}+\gamma v_\pi(S_{t+1})\mid S_t=s]$$

Since the true value function $v_\pi$ is unknown, we replace it with our current estimate $V$ and use an observed transition to construct a sample of the Bellman backup:

$$\mathbb{E}_\pi[Y_t\mid S_t=s]=(T_\pi V)(s)$$

The TD target is therefore not necessarily the true value of $S_t$, but a **sampled estimate** of the Bellman backup of our current value function.

We then incrementally update $V(S_t)$ toward this target, with learning rate $\alpha$:

$$V(S_t)\leftarrow V(S_t)+\alpha[Y_t-V(S_t)]$$

The difference $\delta_t=Y_t-V(S_t)$ is called the **TD error**.

By repeatedly applying these updates, TD learning attempts to make our value estimates satisfy the Bellman expectation equation. Under suitable convergence conditions, tabular TD(0) converges to $v_\pi(s)$.

Unlike Monte Carlo methods, TD does not need to wait until the end of an episode to update its estimates. Both approaches can converge to the same true state-value function, but TD uses one-step bootstrapping whereas MC uses complete sampled returns.

TD(0) resembles DP because both use Bellman equations and bootstrap from current value estimates. However, DP computes expected updates over all possible next states using a known environment model, whereas TD(0) performs sample-based updates using individual observed transitions.
> Sample updates differ from the expected updates
of DP methods in that they are based on a single sample successor rather than on a complete distribution of all possible successors.

TD(0) bases its value-function update in part on an existing estimate $V(S_t)$ which converges towwards $v_\pi(s)$, we say that it is a _bootstrapping method_.

## 6.4 - On-Policy TD Control (SARSA)

SARSA : uses TD-policy for evaluation (prediction)

1. Evaluation (prediction) : estimate the action-state function Q(s,a)

Is it necessary for an on-policy algorithm to learn the action-state function and not only the value function ? It is because they need to compare actions to improve the policy.

Just like TD(0) estimates the state-value function $V^\pi(s)$, SARSA estimates the action-value function $Q^\pi(s,a)$. The main difference is that SARSA updates the value of a state–action pair using the next action selected according to the policy:


$$A_\sim\pi(\cdot|S_t)$$

For a deterministic policy, this simplifies to:


$$A_=\pi(S_t)$$



## Q-learning: Off-policy TD Control

Recall that SARSA uses the update:

$$
Q(S_t,A_t) \leftarrow Q(S_t,A_t)+\alpha[R_{t+1}+\gamma Q(S_{t+1},A_{t+1})-Q(S_t,A_t)]
$$

At each iteration:
- Take action $A_t$ according to the behavior policy and observe $R_{t+1}$ and $S_{t+1}$.
- Sample the next action from the same policy: $A_{t+1}\sim\pi(\cdot|S_{t+1})$.
- Update $Q(S_t,A_t)$ using the value of this sampled next action.

**Q-learning takes a different approach:**

$$
Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha[R_{t+1}+\gamma\max_a Q(S_{t+1},a)-Q(S_t,A_t)] \tag{3}
$$

Instead of using the Q-value of the next action selected by the behavior policy, Q-learning uses the **maximum Q-value over all possible next actions**, regardless of which action is actually taken.

This makes Q-learning **off-policy**, because it learns the value of the greedy target policy while actions can be generated by a different, exploratory behavior policy.

## 6.7 Maximisation Bias and Double Learning

Maximisation Bias : when we select the maximum among several uncertain estimates, we tend to select the one that overestimates the true value.

Problem with Q-learning is that Q-function ($Q(S_t,A_t)$) is used to determine tha maximizing action and to estimate its value which self reinforces the maximisation bias and overestimates the true value.

In **Dual Q-learning** we randomly alternate between $Q_1(a)$, $Q_1(a)$ for uopdate. For example if Q1 is picked then :

1. The next best action is chosen as $$a^* = \underset{a}{\text{argmax }} { Q_1(S_{+1},a)}$$
2. The action-state function Q1 is updated _using $Q_2(s,a)$_ to reduce the maximisation bias. The update rule is as follows. Note the difference with eq. (1) (see previous cell) $$ \boxed{
Q_1(S_t,A_t)\leftarrow Q_1(S_t,A_t)+\alpha
\left[R_{t+1}+\gamma Q_2(S_{t+1},a^*)-Q_1(S_t,A_t)\right]} $$

So when either Q function is chosen at a given iteration, it is used for :
1. Obtaining the next best action
2. Updated using the _other Q function_.

This allows us to unbias estimates.






Policy improvement in dual-Q learning makes the action greedy w.r.t the sum of the two Q functions at a given state S_t that maximises that sum or,

$$a^* = \arg \max_a (Q_1(S_t,a) + Q_2(S_t,a))$$

typical SAC uses 1-step bootstrapping so I skip Chapter 7

You've already covered most of the tabular RL foundations (Chapters 3–6). The next major step is to understand function approximation, actor–critic architectures, and policy gradients

* Chapter 9 — On-policy Prediction with Approximation
> Book pages 197–228 • Essential
> Learn how we replace tabular value functions with parameterized approximations such as neural networks.
> This is the bridge from your Blackjack Q-tables to SAC's neural-network critics.



* Chapter 13 — Policy Gradient Methods
> Book pages 321–337 • Essential
> Learn how to optimize a parameterized policy directly, especially when actions are continuous.
> This introduces the actor–critic framework that SAC builds upon.


* Chapter 11 — Off-policy Methods with Approximation
> Book pages 257–266 • Highly recommended
> Focus on why combining bootstrapping, function approximation, and off-policy learning can cause instability.
> This helps explain some of SAC's design decisions.


Study SAC directly

Move from the textbook to SAC-specific material covering maximum-entropy RL, twin critics, target networks, experience replay, and stochastic continuous-action policies.


## Chapter 9 - On Policy Prediction With Approximation

When the state, action spaces grows large in size tabular methods become computationally and spatially expensive. We therefore want to use approximations, like _interpolations_ ?

The key is _generalization_ : How can experience with a limited subset of the state space be usefully **generalized** to produce a good approximation over a much larger subset?

Methods = function approximation. Its an instance of _supervised learning_. Why ?

Op-policy prediction means estimating the state-value function of policy $\pi$ from experience generated using a known policy $\pi$ (by definition on-policy)
$\pi \rightarrow v_\pi$

Parametrized functional form : the representation of the state is now a linear combination of each state and weight factors. Whereas in tabular methods each state's value was independent from the others, the parametrized functional form allows for generalisation with interdependence

A typical application of parametrized functions is a Deep Neural Network which takes a number of inputs n, returns a number of outputs m >n, and has internal weight matrix **w**.

Parametrized functions mean that at each update more than one state-value may be updated at once, because of interdependence.

In the tabular methods, the learned values at each state (i.e $V_{\pi}(s)$ are decoupled from each other. However in Parametrized functions an update at one state affects many others.

So we define an objective function, **J** which will be a measure of how accurate our model is, or how close from the true state-value function it is, $v_{\pi}$.

Value Error $VE(\vec{w}) = \sum_{s \in S} \mu(s)\left[v_\pi(s)-\hat{v}(s,\vec{w})\right]^2$

But this assumes we know the true value function, but the whole point is to make an estimation of it. How can we get that in the context of RL ?

Since the true value function is unknown, we use sampled returns or bootstrapped estimates as targets, iteratively nudging our predictions toward these targets to approximate the true values



### 9.3 : Stochastic-gradient and Semi-gradient method

Stochastic Gradient Descent (SGD) adjusts the weight vector after each example by a small amount in the direction that would most reduce the error (the negative gradient).

Q : What makes SDG _"stochastic"_ ???

SGD randomly selects one sample $i$ and computes:

$$ \boxed{\mathbf w\leftarrow\mathbf w-\alpha\nabla_{\mathbf w}L_i(\mathbf w)}
$$

Whereas a generic Gradient Descent update will use sa sampled average of the samples. The sample-to-sample based GD makes the gradient vary randomly from update to update.

"Because the true value of a state is the expected value of the return following it, the Monte Carlo target
$U_t \doteq G_t$ is by definition an unbiased estimate of $V_\pi(S_t)$"

The Gradient method defines an arbitrary function (must be differentiable) $\hat{v}(S_t, \mathbf{w}): \mathbb{R}^d \rightarrow \mathbb{R}$ the updates its weights based on Gradient Method.



## Chapter 13 - Gradient Methods

Key insight : We're no longer learning a value function to decide which action to take. Instead, we're directly learning the policy itself.

Instead of learning action values, we define a parameterized policy:

$$\boxed{\pi(a\mid s,\boldsymbol\theta)}$$

Because the end goal is to learn the optimal policy, gradient methods take a "shortcut" compared to action-valued methods.

_"Methods that learn approximations to both policy and value functions are often called actor–critic methods, where ‘actor’ is a reference to the learned policy, and ‘critic’ refers to the learned value function, usually a state-value function"_

Parameterized numerical preferences $h(s,a,\theta) \in \mathbb{R}$ assigns each candidate action a in state s a numerical score called a preference. Uses softmax to convert probability distribution to score.

Q : Why would you want to use action preferences instead of action-values probabilities ? Because preferences make the h converge to 1 i.e policy becomes deterministic over time ?







### Policy Gradient Theorem

We define the performance measure $J(\vec{\theta}) \doteq v_{\pi \theta}(s_0)$. We want to maximise this function, which will in turns maximise the expected return.

Methods that learn a _parametrized policy_ that can select actions without consulting a value function. No more action-value estimates used to decide the action selection.

We define a scalar performance measure $J(\theta)$ such that we want to maximise the performance.

Our goal is therefore to find $\theta^* = \underset{\theta}{\arg \max} J(\theta)$, i.e the parameter $\theta$ that maximises the performance.

$$\theta_{t+1} \leftarrow \theta_{t} + \alpha \nabla J(\theta_t)$$

Methods that learn approximations to both policy and value functions are often called actor–critic methods, where ‘actor’ is a reference to the learned policy, and ‘critic’ refers to the learned value function, usually a state-value function.
> Actor is the learned policy
> Critic is the learn value function


## Actor-Critic Methods

"In actor–critic methods, on the other hand, the state-value function is applied also to the second state of the transition"

The two components are:
- Actor: the policy \(\pi(a\mid s,\theta)\), which selects actions.
- Critic: the value estimate \(\hat v(s,\mathbf w)\), which helps evaluate those actions.

$$
\delta_t=
\underbrace{R_{t+1}+\gamma\hat v(S_{t+1},\mathbf w)}_{\text{reward received + predicted remaining return}}
-
\underbrace{\hat v(S_t,\mathbf w)}_{\text{previous prediction}}.
$$

The CRITIC supplies $\delta_t$ to evaluate the ACTOR

The actor supplies the direction that will increase the likeliness of taking action $A_t$. It _does not_ tell us whether this action is "good" or "bad".

$$
\frac{\nabla_\theta\pi(A_t\mid S_t,\theta_t)}
{\pi(A_t\mid S_t,\theta_t)}
=
\nabla_\theta\log\pi(A_t\mid S_t,\theta_t).
$$

SAC loop :
- Observe state $S_t$
- The actor produces action probabilities, i.e $\pi(a_1 | S_t),...,\pi(a_n | S_t)$
- The actor samples an action from those probabilities.
- Take this sampled action, then Critic computes the feedback $\delta_t$
- Update the actor's parameters.


Update formula :
$$
\theta_{t+1} \doteq \theta_t
+ \alpha(G_{t:t+1}-\hat{v}(S_t,\vec{w}))
\frac{\nabla_\theta\pi(A_t\mid S_t,\theta_t)}
{\pi(A_t\mid S_t,\theta_t)}
$$
+
\(G_{t:t+1}\) denotes the one-step return (expected return by starting at step t in state St and taking action \(A_t\)

## Short review of MC methods and DP algorithms

DP method requires knowledge of the transition probabilities

The value function is defined as the value of starting from a given state and following policy $\pi(s,a)$, or more formally $V^\pi(s)$ is the expected total discounted future reward when starting from $s$ and following $\pi$.

$$
\boxed{V^\pi(s)=\mathbb E_\pi\left[\sum_{t=0}^{\infty}\gamma^t R_{t+1}\mid S_0=s\right]}
$$

Our objective is to find the **best achievable value from every state.**, that is $\boxed{V^*(s)=\max_\pi V^\pi(s)}$

"The Bellman expectation equation evaluates a given policy, whereas the Bellman optimality equation determines the value achievable by following the best possible policy"

